# SMILES label prediction with a classical baseline
This notebook trains class-weighted logistic regression using 2,048-bit Morgan fingerprints. It evaluates an untouched 20% holdout and five-fold cross-validation on the training partition. The rare positive class makes average precision more informative than accuracy. No label meaning is assumed.

Run cells in order. Dependencies: `pandas numpy scipy scikit-learn rdkit nbformat`. Install with `%pip install pandas numpy scipy scikit-learn rdkit nbformat` if needed. Edit `DATA_DIR` for another machine. The source files are never modified.

In [1]:
from pathlib import Path
import json, hashlib, platform
import numpy as np
import pandas as pd
import sklearn, rdkit
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import (average_precision_score, roc_auc_score, accuracy_score,
    balanced_accuracy_score, precision_score, recall_score, f1_score, confusion_matrix)
DATA_DIR = Path.cwd()
if not (DATA_DIR / 'retrospective_revealed_labels.csv').exists():
    DATA_DIR = DATA_DIR / 'data'
SEED = 42
print('Python', platform.python_version(), 'RDKit', rdkit.__version__, 'sklearn', sklearn.__version__)


Python 3.13.3 RDKit 2026.03.6 sklearn 1.9.1


In [2]:
label_path = DATA_DIR / 'retrospective_revealed_labels.csv'
raw = pd.read_csv(label_path)
assert {'CatalogID', 'Label', 'RDKit_SMILES'} <= set(raw.columns)
assert raw.Label.notna().all() and set(raw.Label.unique()) == {0, 1}
print('File SHA256:', hashlib.sha256(label_path.read_bytes()).hexdigest())
print('Raw rows:', len(raw), '\nLabel counts:\n', raw.Label.value_counts().sort_index().to_string())
mols = [Chem.MolFromSmiles(s) if isinstance(s, str) else None for s in raw.RDKit_SMILES]
valid = np.array([m is not None for m in mols])
df = raw.loc[valid].copy()
df['canonical'] = [Chem.MolToSmiles(m, isomericSmiles=True) for m in mols if m is not None]
conflicts = df.groupby('canonical').Label.nunique()
conflicting = set(conflicts[conflicts > 1].index)
conflict_rows = int(df.canonical.isin(conflicting).sum())
df = df.loc[~df.canonical.isin(conflicting)].copy()
duplicates = int(df.canonical.duplicated().sum())
df = df.drop_duplicates('canonical').reset_index(drop=True)
print('Invalid SMILES:', int((~valid).sum()), 'Conflicting-label rows excluded:', conflict_rows,
      'Duplicate structures removed:', duplicates, 'Usable rows:', len(df))
print(df.Label.value_counts().sort_index().to_string())


File SHA256: 55fc2752c75a9aeb1aa0a9b10527768e4ce7f899f82d2efd99b31d41ef9223d5
Raw rows: 6580 
Label counts:
 Label
0    6541
1      39
Invalid SMILES: 0 Conflicting-label rows excluded: 0 Duplicate structures removed: 1 Usable rows: 6579
Label
0    6540
1      39


## Features and evaluation
Morgan radius 2 fingerprints encode local atom environments (including chirality). IDs and labels never enter the feature matrix. Exact canonical duplicates are removed before a stratified 80/20 split (seed 42). Fixed L2 logistic regression uses C=1 and balanced class weights; the decision threshold is fixed at 0.5. There is no tuning against the holdout. Training-only five-fold CV measures variation. This random split can still share related chemical scaffolds, so it does not establish performance on new chemistry.

In [3]:
generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048, includeChirality=True)
def fingerprint(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f'Invalid SMILES: {smiles}')
    return generator.GetFingerprintAsNumPy(mol)
X = np.stack([fingerprint(s) for s in df.canonical])
y = df.Label.to_numpy(dtype=int)
train_idx, test_idx = train_test_split(np.arange(len(y)), test_size=0.2, stratify=y, random_state=SEED)
assert set(df.canonical.iloc[train_idx]).isdisjoint(set(df.canonical.iloc[test_idx]))
X_train, X_test, y_train, y_test = X[train_idx], X[test_idx], y[train_idx], y[test_idx]
print('Train:', len(y_train), 'positives:', int(y_train.sum()))
print('Holdout:', len(y_test), 'positives:', int(y_test.sum()))
def new_model():
    return LogisticRegression(C=1.0, class_weight='balanced', solver='liblinear', max_iter=2000, random_state=SEED)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_scores = cross_validate(new_model(), X_train, y_train, cv=cv,
    scoring={'AP':'average_precision', 'ROC_AUC':'roc_auc', 'balanced_accuracy':'balanced_accuracy'}, n_jobs=1)
cv_table = pd.DataFrame({k[5:]:v for k,v in cv_scores.items() if k.startswith('test_')})
print('Training CV by fold:\n', cv_table.to_string(index=False))
print('CV mean and sample SD:\n', cv_table.agg(['mean','std']).to_string())


Train: 5263 positives: 31
Holdout: 1316 positives: 8
Training CV by fold:
       AP  ROC_AUC  balanced_accuracy
0.339564 0.973416           0.582378
0.660444 0.854982           0.748567
0.719202 0.992762           0.927137
0.555064 0.863448           0.832377
0.223789 0.860102           0.580943
CV mean and sample SD:
             AP   ROC_AUC  balanced_accuracy
mean  0.499613  0.908942           0.734281
std   0.211457  0.068098           0.152977


In [4]:
model = new_model().fit(X_train, y_train)
dummy = DummyClassifier(strategy='prior').fit(X_train, y_train)
def evaluate(estimator):
    p = estimator.predict_proba(X_test)[:, 1]
    pred = (p >= 0.5).astype(int)
    return {'Average precision':average_precision_score(y_test,p), 'ROC AUC':roc_auc_score(y_test,p),
        'Accuracy':accuracy_score(y_test,pred), 'Balanced accuracy':balanced_accuracy_score(y_test,pred),
        'Precision':precision_score(y_test,pred,zero_division=0), 'Recall':recall_score(y_test,pred),
        'F1':f1_score(y_test,pred,zero_division=0)}, p, pred
metrics, probabilities, predictions = evaluate(model)
baseline_metrics, _, _ = evaluate(dummy)
results = pd.DataFrame({'Logistic regression':metrics, 'Prior baseline':baseline_metrics})
cm = confusion_matrix(y_test, predictions, labels=[0,1])
print(results.to_string(float_format=lambda v:f'{v:.4f}'))
print('Confusion matrix (rows true 0/1; columns predicted 0/1):\n', cm)
print('Holdout positive prevalence:', y_test.mean())
# Holdout predictions remain from the training-only model, before the final refit.
holdout_predictions = df.iloc[test_idx][['CatalogID','RDKit_SMILES','Label']].copy()
holdout_predictions['score'] = probabilities
holdout_predictions['predicted_label'] = predictions
holdout_predictions.sort_values('score', ascending=False).head(10)


                   Logistic regression  Prior baseline
Average precision               0.2676          0.0061
ROC AUC                         0.8831          0.5000
Accuracy                        0.9924          0.9939
Balanced accuracy               0.6235          0.5000
Precision                       0.3333          0.0000
Recall                          0.2500          0.0000
F1                              0.2857          0.0000
Confusion matrix (rows true 0/1; columns predicted 0/1):
 [[1304    4]
 [   6    2]]
Holdout positive prevalence: 0.0060790273556231


,CatalogID,RDKit_SMILES,Label,score,predicted_label
15,Z359457804,O=C(Nc1cccc(-c2ccn[nH]2)c1)c1ccc2ccccc2n1,1,0.946267,1
3682,Z2125630906,COc1c(C(=O)Nc2[nH]nc(C)c2C#N)cnc2c1cnn2C,0,0.903056,1
35,Z1445518982,O=C(O)CCc1ccc(NC(=O)c2cccc3cn[nH]c23)cc1,1,0.864735,1
1335,Z442962706,O=C(Nc1cccc(C#Cc2ccccn2)c1)c1ccc(-c2ccccc2)[nH...,0,0.632243,1
619,Z788118638,N#Cc1c(NC(=O)Nc2cccnc2)sc2c1CCNC2,0,0.610285,1
2699,Z27033169,O=C(Nc1nc2c(F)c(F)c(F)cc2s1)c1ccc(Cl)cc1[N+](=...,0,0.586646,1
5007,Z1603682130,O=C(O)CC(NC(=O)c1cccc2cn[nH]c12)c1ccc(Cl)c(F)c1,0,0.451474,0
3676,Z2129063473,O=C(Nc1cccnc1)c1ccnc(-c2ccc(Cl)cc2)n1,0,0.432816,0
3014,Z240721912,C#Cc1cccc(NC(=O)c2ccc(=O)[nH]n2)c1,0,0.428337,0
4008,Z19739821,N#Cc1ccc(OCC(=O)Nc2nc3ccc(F)cc3s2)cc1,0,0.384625,0


## Robustness checks: random versus scaffold-held-out validation
There are only 39 positive molecules, each on a different Bemis-Murcko scaffold. Average precision (AP) summarizes precision-recall performance across score thresholds; precision@50 is the fraction of the ranked top 50 that are hits, and recall@50 is the fraction of validation hits captured in that shortlist. These ranking measures, not accuracy at an arbitrary threshold, are the main measures here. The single 20% holdout contains just 8 positives and can give a noisy estimate. The following checks keep the baseline model and parameters fixed: 30 stratified random 80/20 splits show sensitivity to which molecules are held out, while five-fold scaffold-grouped CV tests generalization to scaffolds absent from training. These are diagnostic estimates, not leaderboard scores or formal confidence intervals; the random folds share chemistry and are expected to be optimistic. Top-50 counts refer to the highest-scored 50 molecules within each validation fold.

In [5]:
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import StratifiedGroupKFold

scaffolds = np.array([
    MurckoScaffold.MurckoScaffoldSmiles(
        mol=Chem.MolFromSmiles(smiles), includeChirality=True
    )
    for smiles in df.canonical
])
positive_scaffolds = set(scaffolds[y == 1])
print('Usable molecules:', len(y))
print('Positive molecules:', int(y.sum()))
print('Distinct positive scaffolds:', len(positive_scaffolds))
print('All distinct scaffolds:', len(set(scaffolds)))

def evaluate_split(train, valid):
    estimator = new_model().fit(X[train], y[train])
    scores = estimator.predict_proba(X[valid])[:, 1]
    top50 = np.argsort(scores)[::-1][:min(50, len(valid))]
    hits_top50 = int(y[valid][top50].sum())
    positives = int(y[valid].sum())
    return {
        'AP': average_precision_score(y[valid], scores),
        'ROC_AUC': roc_auc_score(y[valid], scores),
        'Top50_hits': hits_top50,
        'Precision_at_50': hits_top50 / len(top50),
        'Recall_at_50': hits_top50 / positives if positives else np.nan,
        'Validation_positives': positives,
        'Validation_molecules': len(valid),
    }

random_results = []
for split_seed in range(30):
    train, valid = train_test_split(
        np.arange(len(y)), test_size=0.2, stratify=y, random_state=split_seed
    )
    random_results.append(evaluate_split(train, valid))

random_results = pd.DataFrame(random_results)
print('\nRepeated stratified random 80/20 splits (30 seeds):')
ranking_metrics = ['AP', 'ROC_AUC', 'Top50_hits', 'Precision_at_50', 'Recall_at_50']
print(random_results[ranking_metrics].agg(['mean', 'std']).to_string())
print('AP quantiles:')
print(random_results.AP.quantile([0, 0.1, 0.25, 0.5, 0.75, 0.9, 1]).to_string())
print('Top-50 hit-count range:', int(random_results.Top50_hits.min()), 'to',
      int(random_results.Top50_hits.max()))
print('Expected random top-50 hits at validation prevalence:',
      f"{(50 * random_results.Validation_positives / random_results.Validation_molecules).mean():.2f}")

holdout_top50 = np.argsort(probabilities)[::-1][:50]
holdout_hits = int(y_test[holdout_top50].sum())
print('\nOriginal untouched holdout ranking (top 50 of 1,316):')
print(f'Hits: {holdout_hits}/50; precision@50: {holdout_hits / len(holdout_top50):.3f}; '
      f'recall@50: {holdout_hits / int(y_test.sum()):.3f}; '
      f'random expected hits: {50 * y_test.mean():.2f}')

scaffold_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
scaffold_results = []
for fold, (train, valid) in enumerate(
    scaffold_cv.split(X, y, groups=scaffolds), start=1
):
    overlap = set(scaffolds[train]) & set(scaffolds[valid])
    assert not overlap, f'Scaffold leakage in fold {fold}: {len(overlap)} shared scaffolds'
    result = evaluate_split(train, valid)
    result['Fold'] = fold
    scaffold_results.append(result)

scaffold_results = pd.DataFrame(scaffold_results).set_index('Fold')
print('\nFive-fold scaffold-held-out CV (no scaffold overlap):')
print(scaffold_results.to_string(float_format=lambda value: f'{value:.3f}'))
print('Mean and fold SD:')
print(scaffold_results[ranking_metrics].agg(['mean', 'std']).to_string())


Usable molecules: 6579
Positive molecules: 39
Distinct positive scaffolds: 39
All distinct scaffolds: 5035

Repeated stratified random 80/20 splits (30 seeds):
            AP   ROC_AUC  Top50_hits  Precision_at_50  Recall_at_50
mean  0.479263  0.902832    5.566667         0.111333      0.695833
std   0.206469  0.069049    1.546594         0.030932      0.193324
AP quantiles:
0.00    0.056272
0.10    0.190005
0.25    0.409014
0.50    0.473071
0.75    0.610589
0.90    0.704058
1.00    0.860841
Top-50 hit-count range: 2 to 8
Expected random top-50 hits at validation prevalence: 0.30

Original untouched holdout ranking (top 50 of 1,316):
Hits: 4/50; precision@50: 0.080; recall@50: 0.500; random expected hits: 0.30

Five-fold scaffold-held-out CV (no scaffold overlap):
        AP  ROC_AUC  Top50_hits  Precision_at_50  Recall_at_50  Validation_positives  Validation_molecules
Fold                                                                                                      
1    0.547 

## Rank unlabeled candidates for review
The candidate file overlaps the revealed-label file, so this cell excludes rows whose CatalogID or canonical structure is already labeled. It compares the Morgan-only logistic baseline with fixed alternatives that add RDKit physicochemical descriptors (scaled inside each training fold) and nonlinear ExtraTrees, plus nearest-positive similarity and a pre-specified rank blend. All models use identical scaffold-held-out folds; no candidate labels are used. A challenger replaces the baseline ranking only if its mean scaffold-CV AP and mean hits@50 both exceed the baseline; this conservative rule is still exploratory model selection, not an unbiased performance estimate. With only 39 positive examples, the baseline remains the fallback. Scores are ranking scores, not calibrated probabilities. Outputs are retrospective review files, not challenge submissions. Each run appends model evaluations and ranking-artifact hashes to `experiment_submission_log.csv`. A separate diversity-oriented shortlist greedily spreads picks across the selected model's top 500 candidates.

In [6]:
from rdkit import DataStructs
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem import Descriptors, Lipinski, MACCSkeys, rdMolDescriptors
from datetime import datetime, timezone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

descriptor_names = [
    'MolecularWeight', 'RDKit_cLogP', 'TPSA_A2', 'HBD', 'HBA',
    'RotatableBonds', 'HeavyAtomCount', 'RingCount', 'AromaticRingCount',
    'FractionCSP3', 'FormalCharge'
]
def physicochemical_features(mol):
    return [
        Descriptors.MolWt(mol), Descriptors.MolLogP(mol), Descriptors.TPSA(mol),
        Lipinski.NumHDonors(mol), Lipinski.NumHAcceptors(mol),
        Lipinski.NumRotatableBonds(mol), Lipinski.HeavyAtomCount(mol),
        rdMolDescriptors.CalcNumRings(mol), rdMolDescriptors.CalcNumAromaticRings(mol),
        rdMolDescriptors.CalcFractionCSP3(mol),
        sum(atom.GetFormalCharge() for atom in mol.GetAtoms()),
    ]
def bitvectors_to_array(fingerprints):
    matrix = np.zeros((len(fingerprints), fingerprints[0].GetNumBits()), dtype=np.uint8)
    for row, fingerprint in enumerate(fingerprints):
        DataStructs.ConvertToNumpyArray(fingerprint, matrix[row])
    return matrix

def extract_all_features(smiles_list, feature_names=None):
    available_features = {'morgan', 'maccs', 'torsion', 'physchem'}
    requested = available_features if feature_names is None else set(feature_names)
    unknown = requested - available_features
    if unknown:
        raise ValueError(f'Unknown feature names: {sorted(unknown)}')
    if not smiles_list:
        raise ValueError('At least one SMILES string is required')
    molecules = [Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None
                 for smiles in smiles_list]
    invalid = [index for index, molecule in enumerate(molecules) if molecule is None]
    if invalid:
        raise ValueError(f'Invalid SMILES at input rows: {invalid[:10]}')
    features = {}
    morgan_fps = []
    if 'morgan' in requested:
        generator = rdFingerprintGenerator.GetMorganGenerator(
            radius=2, fpSize=2048, includeChirality=True
        )
        morgan_fps = [generator.GetFingerprint(molecule) for molecule in molecules]
        features['morgan'] = bitvectors_to_array(morgan_fps)
    if 'maccs' in requested:
        features['maccs'] = bitvectors_to_array([
            MACCSkeys.GenMACCSKeys(molecule) for molecule in molecules
        ])
    if 'torsion' in requested:
        generator = rdFingerprintGenerator.GetTopologicalTorsionGenerator(fpSize=2048)
        features['torsion'] = bitvectors_to_array([
            generator.GetFingerprint(molecule) for molecule in molecules
        ])
    if 'physchem' in requested:
        features['physchem'] = np.asarray(
            [physicochemical_features(molecule) for molecule in molecules], dtype=np.float32
        )
    return molecules, features, morgan_fps

labeled_mols, labeled_features, labeled_fps = extract_all_features(df.canonical.tolist())
assert np.array_equal(labeled_features['morgan'], X), 'Shared Morgan features differ from baseline features'
X = labeled_features['morgan']
maccs_X = labeled_features['maccs']
torsion_X = labeled_features['torsion']
physchem_X = labeled_features['physchem']
assert physchem_X.shape == (len(df), len(descriptor_names))
X_augmented = np.column_stack((X.astype(np.float32), physchem_X))
fp_columns = list(range(X.shape[1]))
descriptor_columns = list(range(X.shape[1], X_augmented.shape[1]))
def new_augmented_logistic():
    scale_descriptors = ColumnTransformer([
        ('fingerprints', 'passthrough', fp_columns),
        ('physicochemical', StandardScaler(), descriptor_columns),
    ], remainder='drop')
    return make_pipeline(scale_descriptors, LogisticRegression(
        C=1.0, class_weight='balanced', solver='liblinear', max_iter=2000,
        random_state=SEED,
    ))
def new_extra_trees():
    return ExtraTreesClassifier(
        n_estimators=300, min_samples_leaf=2, max_features=0.2,
        class_weight='balanced', n_jobs=-1, random_state=SEED,
    )
candidate_path = DATA_DIR / 'retrospective_split_nolabels.csv'
candidate_raw = pd.read_csv(candidate_path)
assert {'CatalogID', 'SMILES'} <= set(candidate_raw.columns)
assert candidate_raw.CatalogID.notna().all() and candidate_raw.CatalogID.is_unique
candidate_mols = [Chem.MolFromSmiles(s) if isinstance(s, str) else None
                  for s in candidate_raw.SMILES]
invalid_candidates = sum(mol is None for mol in candidate_mols)
assert invalid_candidates == 0, f'{invalid_candidates} invalid candidate SMILES'
candidate_canonical = [Chem.MolToSmiles(mol, isomericSmiles=True)
                      for mol in candidate_mols]
known_ids = set(raw.CatalogID.astype(str))
known_structures = set(df.canonical)
known_id_mask = candidate_raw.CatalogID.astype(str).isin(known_ids).to_numpy()
known_structure_mask = np.isin(candidate_canonical, list(known_structures))
exclude_mask = known_id_mask | known_structure_mask
print('Candidate rows:', len(candidate_raw))
print('Known CatalogID overlaps:', int(known_id_mask.sum()))
print('Known exact-structure overlaps:', int(known_structure_mask.sum()))
print('Excluded known rows (union):', int(exclude_mask.sum()))
candidate_indices = np.flatnonzero(~exclude_mask)
candidate_ids = candidate_raw.CatalogID.iloc[candidate_indices].astype(str).to_numpy()
candidate_smiles = candidate_raw.SMILES.iloc[candidate_indices].to_numpy()
candidate_mols = [candidate_mols[i] for i in candidate_indices]
candidate_canonical = [candidate_canonical[i] for i in candidate_indices]
assert len(set(candidate_canonical)) == len(candidate_canonical), 'Duplicate candidate structures remain'

similarity_cv_rows = []
for fold, (train, valid) in enumerate(
    scaffold_cv.split(X, y, groups=scaffolds), start=1
):
    fold_model = new_model().fit(X[train], y[train])
    fold_scores = fold_model.predict_proba(X[valid])[:, 1]
    fold_augmented_logistic = new_augmented_logistic().fit(X_augmented[train], y[train])
    augmented_logistic_scores = fold_augmented_logistic.predict_proba(X_augmented[valid])[:, 1]
    fold_extra_trees = new_extra_trees().fit(X_augmented[train], y[train])
    extra_trees_scores = fold_extra_trees.predict_proba(X_augmented[valid])[:, 1]
    fold_positive_fps = [labeled_fps[i] for i in train if y[i] == 1]
    similarity_scores = np.array([
        max(DataStructs.BulkTanimotoSimilarity(labeled_fps[i], fold_positive_fps))
        for i in valid
    ])
    score_ranks = pd.Series(fold_scores).rank(pct=True).to_numpy()
    similarity_ranks = pd.Series(similarity_scores).rank(pct=True).to_numpy()
    fold_methods = {
        'Logistic baseline': fold_scores,
        'Morgan plus physicochemical logistic': augmented_logistic_scores,
        'Morgan plus physicochemical ExtraTrees': extra_trees_scores,
        'Nearest-positive similarity': similarity_scores,
        '50/50 rank average': (score_ranks + similarity_ranks) / 2,
    }
    for method, scores in fold_methods.items():
        top = np.argsort(scores)[::-1][:50]
        hits = int(y[valid][top].sum())
        positives = int(y[valid].sum())
        similarity_cv_rows.append({
            'Fold': fold,
            'Method': method,
            'AP': average_precision_score(y[valid], scores),
            'Hits_at_50': hits,
            'Precision_at_50': hits / len(top),
            'Recall_at_50': hits / positives,
        })
similarity_cv = pd.DataFrame(similarity_cv_rows)
print('\nScaffold-held-out comparison (mean and fold SD):')
print(similarity_cv.groupby('Method')[
    ['AP', 'Hits_at_50', 'Precision_at_50', 'Recall_at_50']
].agg(['mean', 'std']).to_string(float_format=lambda value: f'{value:.3f}'))
cv_means = similarity_cv.groupby('Method')[['AP', 'Hits_at_50']].mean()
baseline_mean = cv_means.loc['Logistic baseline']
eligible_challengers = [
    name for name in ('Morgan plus physicochemical logistic',
                      'Morgan plus physicochemical ExtraTrees')
    if cv_means.loc[name, 'AP'] > baseline_mean.AP
    and cv_means.loc[name, 'Hits_at_50'] > baseline_mean.Hits_at_50
]
if eligible_challengers:
    selected_model_name = max(eligible_challengers, key=lambda name: cv_means.loc[name, 'AP'])
else:
    selected_model_name = 'Logistic baseline'
print('\nSelected ranking model under the predeclared AP + hits@50 rule:', selected_model_name)
print('This selection uses the same small CV set for comparison; metrics are exploratory, not an unbiased post-selection estimate.')

candidate_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2, fpSize=2048, includeChirality=True
)
candidate_fps = [candidate_generator.GetFingerprint(mol) for mol in candidate_mols]
known_hit_fps = [labeled_fps[i] for i in np.flatnonzero(y == 1)]
if selected_model_name == 'Morgan plus physicochemical logistic':
    full_model = new_augmented_logistic().fit(X_augmented, y)
elif selected_model_name == 'Morgan plus physicochemical ExtraTrees':
    full_model = new_extra_trees().fit(X_augmented, y)
else:
    full_model = new_model().fit(X, y)
candidate_scores = np.empty(len(candidate_mols), dtype=float)
batch_size = 5000
for start in range(0, len(candidate_mols), batch_size):
    stop = min(start + batch_size, len(candidate_mols))
    requested_candidate_features = (
        ['morgan'] if selected_model_name == 'Logistic baseline'
        else ['morgan', 'physchem']
    )
    _, candidate_batch_features, _ = extract_all_features(
        candidate_smiles[start:stop].tolist(),
        feature_names=requested_candidate_features,
    )
    batch_X = candidate_batch_features['morgan']
    if selected_model_name == 'Logistic baseline':
        model_batch_X = batch_X
    else:
        model_batch_X = np.column_stack((
            batch_X.astype(np.float32), candidate_batch_features['physchem']
        ))
    candidate_scores[start:stop] = full_model.predict_proba(model_batch_X)[:, 1]
ranked = pd.DataFrame({
    'CandidateIndex': np.arange(len(candidate_ids)),
    'CatalogID': candidate_ids,
    'SMILES': candidate_smiles,
    'ModelScore': candidate_scores,
    'RankingModel': selected_model_name,
    'CanonicalSMILES': candidate_canonical,
})
ranked = ranked.sort_values(
    ['ModelScore', 'CatalogID'], ascending=[False, True], kind='stable'
).reset_index(drop=True)
ranked.insert(0, 'BaselineRank', np.arange(1, len(ranked) + 1))
ranked_path = DATA_DIR / 'retrospective_candidates_ranked.csv'
ranked.drop(columns=['CandidateIndex', 'CanonicalSMILES']).to_csv(ranked_path, index=False)

top_review = ranked.head(500).copy()
top_review_indices = top_review.CandidateIndex.to_numpy()
top_review_fps = [candidate_fps[i] for i in top_review_indices]
top_review_similarity = np.array([
    max(DataStructs.BulkTanimotoSimilarity(fp, known_hit_fps))
    for fp in top_review_fps
])
top_review['MaxTanimotoToKnownHit'] = top_review_similarity
top50_model = top_review.head(50).copy()
top50_model.to_csv(DATA_DIR / 'retrospective_top50_model_ranked.csv', index=False)

selected = [0]
max_similarity_to_selected = np.zeros(len(top_review), dtype=float)
for _ in range(49):
    last_fp = top_review_fps[selected[-1]]
    similarities = np.asarray(DataStructs.BulkTanimotoSimilarity(last_fp, top_review_fps))
    max_similarity_to_selected = np.maximum(max_similarity_to_selected, similarities)
    available = np.ones(len(top_review), dtype=bool)
    available[selected] = False
    remaining = np.flatnonzero(available)
    next_local = remaining[np.lexsort((
        -top_review.ModelScore.to_numpy()[remaining],
        max_similarity_to_selected[remaining],
    ))[0]]
    selected.append(int(next_local))
diverse50 = top_review.iloc[selected].copy()
diverse50.insert(0, 'DiversityPickOrder', np.arange(1, len(diverse50) + 1))
diverse50.to_csv(DATA_DIR / 'retrospective_top50_diverse_review.csv', index=False)

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S.%fZ')
label_sha256 = sha256_file(label_path)
candidate_sha256 = sha256_file(candidate_path)
model_summary = similarity_cv.groupby('Method')[
    ['AP', 'Hits_at_50', 'Precision_at_50', 'Recall_at_50']
].agg(['mean', 'std'])
model_configurations = {
    'Logistic baseline': '2048-bit chirality-aware Morgan radius-2 fingerprints; class-weighted L2 logistic regression; C=1.',
    'Morgan plus physicochemical logistic': 'Baseline Morgan bits plus 11 RDKit descriptors; descriptor StandardScaler fit on each training fold; class-weighted L2 logistic regression; C=1.',
    'Morgan plus physicochemical ExtraTrees': 'Baseline Morgan bits plus 11 RDKit descriptors; ExtraTrees with 300 trees, min_samples_leaf=2, max_features=0.2, class_weight=balanced.',
    'Nearest-positive similarity': 'Maximum Morgan radius-2, 2048-bit chirality-aware Tanimoto similarity to training positives.',
    '50/50 rank average': 'Mean percentile rank of baseline logistic score and nearest-positive Morgan similarity.',
}
ledger_rows = []
for method, values in model_summary.iterrows():
    metric_summary = {
        metric: {stat: float(values[(metric, stat)]) for stat in ('mean', 'std')}
        for metric in ('AP', 'Hits_at_50', 'Precision_at_50', 'Recall_at_50')
    }
    metric_summary['model_configuration'] = model_configurations[method]
    ledger_rows.append({
        'RunID': run_id,
        'RecordType': 'model_evaluation',
        'RecordID': method,
        'Status': 'evaluated_on_scaffold_held_out_folds',
        'LabelDataSHA256': label_sha256,
        'CandidateDataSHA256': candidate_sha256,
        'ModelOrSelection': method,
        'ParametersOrMetrics': json.dumps(metric_summary, sort_keys=True),
        'ArtifactPath': '',
        'ArtifactSHA256': '',
        'SubmissionReference': '',
        'Notes': 'Fixed methods; validation grouped by Bemis-Murcko scaffold; no hidden labels used.',
    })
artifact_specs = [
    ('full_ranked_candidates', ranked_path, f'Candidate ranking model: {selected_model_name}.'),
    ('model_top50_review', DATA_DIR / 'retrospective_top50_model_ranked.csv',
     'Exploratory shortlist, not submitted.'),
    ('diverse_top50_review', DATA_DIR / 'retrospective_top50_diverse_review.csv',
     'Diversity-oriented exploratory shortlist, not submitted.'),
]
for record_id, artifact_path, notes in artifact_specs:
    ledger_rows.append({
        'RunID': run_id,
        'RecordType': 'ranking_artifact',
        'RecordID': record_id,
        'Status': 'generated_for_review',
        'LabelDataSHA256': label_sha256,
        'CandidateDataSHA256': candidate_sha256,
        'ModelOrSelection': selected_model_name,
        'ParametersOrMetrics': json.dumps({
            'model_configuration': model_configurations[selected_model_name],
            'rows': int(len(pd.read_csv(artifact_path))),
            'selected_model_scaffold_cv_ap_mean': float(cv_means.loc[selected_model_name, 'AP']),
            'selected_model_scaffold_cv_hits_at_50_mean': float(cv_means.loc[selected_model_name, 'Hits_at_50']),
            'Morgan_radius': 2, 'Morgan_bits': 2048, 'include_chirality': True,
            'physicochemical_descriptors': descriptor_names if selected_model_name != 'Logistic baseline' else [],
            'seed': SEED,
        }, sort_keys=True),
        'ArtifactPath': artifact_path.name,
        'ArtifactSHA256': sha256_file(artifact_path),
        'SubmissionReference': '',
        'Notes': notes,
    })
ledger_path = DATA_DIR / 'experiment_submission_log.csv'
new_ledger = pd.DataFrame(ledger_rows)
if ledger_path.exists():
    new_ledger = pd.concat([pd.read_csv(ledger_path), new_ledger], ignore_index=True)
new_ledger.to_csv(ledger_path, index=False)
print('Experiment/submission ledger appended:', ledger_path)
print('Ledger run ID:', run_id, '| model/ranking records added:', len(ledger_rows))

def diversity_summary(frame, fps):
    sims = [
        DataStructs.TanimotoSimilarity(fps[i], fps[j])
        for i in range(len(fps)) for j in range(i)
    ]
    murcko = [MurckoScaffold.MurckoScaffoldSmiles(
        mol=Chem.MolFromSmiles(s), includeChirality=True
    ) for s in frame.SMILES]
    return len(set(murcko)), float(np.mean(sims)) if sims else np.nan

top50_fps = top_review_fps[:50]
diverse_fps = [top_review_fps[i] for i in selected]
ranked_scaffolds, ranked_pair_similarity = diversity_summary(top50_model, top50_fps)
diverse_scaffolds, diverse_pair_similarity = diversity_summary(diverse50, diverse_fps)
print('\nScored unlabeled candidates:', len(ranked))
print('Ranking model:', selected_model_name)
print('Full ranking saved to:', ranked_path)
print('Model top-50 saved to:', DATA_DIR / 'retrospective_top50_model_ranked.csv')
print('Diverse review top-50 saved to:', DATA_DIR / 'retrospective_top50_diverse_review.csv')
print('\nTop 20 model-ranked candidates (score is not a calibrated probability):')
print(top_review[['BaselineRank', 'CatalogID', 'ModelScore', 'MaxTanimotoToKnownHit']]
      .head(20).to_string(index=False, float_format=lambda value: f'{value:.3f}'))
print('\nTop-50 diversity comparison:')
print(f'Model-ranked: {ranked_scaffolds} distinct Murcko scaffolds; '
      f'mean pairwise Tanimoto={ranked_pair_similarity:.3f}')
print(f'Diversity-oriented: {diverse_scaffolds} distinct Murcko scaffolds; '
      f'mean pairwise Tanimoto={diverse_pair_similarity:.3f}')
assert len(top50_model) == 50 and len(diverse50) == 50


Candidate rows: 184632
Known CatalogID overlaps: 6580
Known exact-structure overlaps: 6573
Excluded known rows (union): 6583

Scaffold-held-out comparison (mean and fold SD):
                                          AP       Hits_at_50       Precision_at_50       Recall_at_50      
                                        mean   std       mean   std            mean   std         mean   std
Method                                                                                                      
50/50 rank average                     0.372 0.204      4.800 1.643           0.096 0.033        0.621 0.229
Logistic baseline                      0.421 0.199      5.400 1.517           0.108 0.030        0.693 0.188
Morgan plus physicochemical ExtraTrees 0.302 0.210      4.000 2.121           0.080 0.042        0.514 0.267
Morgan plus physicochemical logistic   0.373 0.234      5.000 1.581           0.100 0.032        0.639 0.189
Nearest-positive similarity            0.313 0.215      4.400 

## Optional practical screen and QSVC audit
This notebook is the single source of truth for candidate scores, the full ranked CSV, and both top-50 lists. After running the ranking/export cell above, optionally run `python data/generate_qsvc_audit.py` from the repository root to create only `retrospective_practical_screen.csv` and `retrospective_top500_qsvc_audit.csv`. The script reads `retrospective_candidates_ranked.csv`; it does not retrain the ranking model or rewrite any ranking/top-50 artifacts. The practical screen is a review aid, not a verified prospective submission list.

## Experimental quantum model: small VQC, not an assumed improvement
This optional experiment compares a four-qubit PennyLane variational quantum classifier (VQC) with the classical baseline under the same scaffold-held-out folds. Each fold fits Morgan-fingerprint compression and descriptor scaling on its training partition only, then tests 1:5 and 1:10 positive-to-negative undersampling of training negatives; validation keeps the natural class prevalence. The VQC is deliberately small because only 39 positives are available. This is a classical simulation, not evidence of quantum advantage. Qiskit Aer separately benchmarks a matched circuit's transpiled depth and synthetic depolarizing-noise sensitivity on a small validation sample; those noise rates are illustrative, not calibrated to hardware or assay uncertainty. This experiment does not replace the full-catalog ranking: only 500 candidate compounds are currently reviewed, and the VQC has not been validated as a scalable scoring method for all 178k candidates.

In [7]:
import pennylane as qml
from pennylane import numpy as pnp
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.preprocessing import StandardScaler
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

N_QUBITS = 4
VQC_LAYERS = 2
VQC_EPOCHS = 20
VQC_SHOTS = 2048
vqc_device = qml.device('default.qubit', wires=N_QUBITS)

@qml.qnode(vqc_device, interface='autograd', diff_method='backprop')
def vqc_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(N_QUBITS), rotation='Y')
    for layer in range(VQC_LAYERS):
        for wire in range(N_QUBITS):
            qml.RY(weights[layer, wire, 0], wires=wire)
            qml.RZ(weights[layer, wire, 1], wires=wire)
        for wire in range(N_QUBITS):
            qml.CNOT(wires=[wire, (wire + 1) % N_QUBITS])
    return qml.expval(qml.PauliZ(0))

def prepare_vqc_features(train_idx, valid_idx):
    fingerprint_svd = TruncatedSVD(n_components=4, n_iter=5, random_state=SEED)
    train_fp = fingerprint_svd.fit_transform(X[train_idx])
    valid_fp = fingerprint_svd.transform(X[valid_idx])
    descriptor_scaler = StandardScaler().fit(physchem_X[train_idx])
    train_desc = descriptor_scaler.transform(physchem_X[train_idx])
    valid_desc = descriptor_scaler.transform(physchem_X[valid_idx])
    train_combined = np.column_stack((train_fp, train_desc))
    valid_combined = np.column_stack((valid_fp, valid_desc))
    projection = PCA(n_components=N_QUBITS, random_state=SEED).fit(train_combined)
    train_projected = projection.transform(train_combined)
    valid_projected = projection.transform(valid_combined)
    angle_scaler = StandardScaler().fit(train_projected)
    train_angles = np.pi * np.tanh(angle_scaler.transform(train_projected) / 2.0)
    valid_angles = np.pi * np.tanh(angle_scaler.transform(valid_projected) / 2.0)
    return train_angles.astype(np.float64), valid_angles.astype(np.float64)

def fit_vqc(train_angles, train_labels, seed):
    rng = np.random.default_rng(seed)
    weights = pnp.array(
        rng.uniform(-0.05, 0.05, size=(VQC_LAYERS, N_QUBITS, 2)),
        requires_grad=True,
    )
    angles = pnp.array(train_angles, requires_grad=False)
    targets_z = pnp.array(1.0 - 2.0 * train_labels, requires_grad=False)
    optimizer = qml.AdamOptimizer(stepsize=0.03)
    def objective(parameters):
        return pnp.mean((vqc_circuit(angles, parameters) - targets_z) ** 2)
    for _ in range(VQC_EPOCHS):
        weights = optimizer.step(objective, weights)
    return weights

def predict_vqc(weights, angles, batch_size=128):
    probabilities = []
    for start in range(0, len(angles), batch_size):
        z_expectation = np.asarray(vqc_circuit(
            pnp.array(angles[start:start + batch_size], requires_grad=False), weights
        ), dtype=float).reshape(-1)
        probabilities.extend(((1.0 - z_expectation) / 2.0).tolist())
    return np.clip(np.asarray(probabilities), 0.0, 1.0)

vqc_cv_rows = []
noise_benchmark = None
for fold, (train_idx, valid_idx) in enumerate(
    scaffold_cv.split(X, y, groups=scaffolds), start=1
):
    train_angles, valid_angles = prepare_vqc_features(train_idx, valid_idx)
    positive_train = train_idx[y[train_idx] == 1]
    negative_train = train_idx[y[train_idx] == 0]
    for ratio in (5, 10):
        rng = np.random.default_rng(SEED + fold * 100 + ratio)
        sampled_negative = rng.choice(
            negative_train,
            size=min(len(negative_train), ratio * len(positive_train)),
            replace=False,
        )
        sampled_train = np.concatenate((positive_train, sampled_negative))
        local_train = np.searchsorted(train_idx, sampled_train)
        local_order = rng.permutation(len(local_train))
        local_train = local_train[local_order]
        weights = fit_vqc(
            train_angles[local_train], y[sampled_train][local_order],
            seed=SEED + fold * 100 + ratio,
        )
        scores = predict_vqc(weights, valid_angles)
        top = np.argsort(scores)[::-1][:min(50, len(valid_idx))]
        hits = int(y[valid_idx][top].sum())
        positives = int(y[valid_idx].sum())
        vqc_cv_rows.append({
            'Fold': fold,
            'Method': f'PennyLane VQC 1:{ratio}',
            'Training_positives': int(y[sampled_train].sum()),
            'Training_negatives': int((y[sampled_train] == 0).sum()),
            'AP': average_precision_score(y[valid_idx], scores),
            'Hits_at_50': hits,
            'Precision_at_50': hits / len(top),
            'Recall_at_50': hits / positives,
        })
        if fold == 1 and ratio == 5:
            noise_benchmark = (weights, valid_angles, valid_idx, scores)

vqc_cv = pd.DataFrame(vqc_cv_rows)
print('PennyLane VQC scaffold-held-out results (validation prevalence unchanged):')
print(vqc_cv.groupby('Method')[
    ['AP', 'Hits_at_50', 'Precision_at_50', 'Recall_at_50']
].agg(['mean', 'std']).to_string(float_format=lambda value: f'{value:.3f}'))
baseline_cv = similarity_cv.loc[similarity_cv.Method.eq('Logistic baseline')]
baseline_ap = float(baseline_cv.AP.mean())
baseline_hits = float(baseline_cv.Hits_at_50.mean())
vqc_summary = vqc_cv.groupby('Method')[['AP', 'Hits_at_50']].mean()
for method, row in vqc_summary.iterrows():
    print(f'{method}: AP delta vs baseline={row.AP - baseline_ap:+.3f}; '
          f'hits@50 delta={row.Hits_at_50 - baseline_hits:+.2f}')

if noise_benchmark is None:
    raise RuntimeError('The VQC noise benchmark weights were not produced.')
noise_weights, noise_angles, noise_valid_idx, _ = noise_benchmark
noise_rng = np.random.default_rng(SEED)
sample_count = min(64, len(noise_valid_idx))
noise_positions = np.sort(noise_rng.choice(len(noise_valid_idx), sample_count, replace=False))
noise_circuits = []
noise_reference = []
for position in noise_positions:
    circuit = QuantumCircuit(N_QUBITS, 1)
    for wire in range(N_QUBITS):
        circuit.ry(float(noise_angles[position, wire]), wire)
    for layer in range(VQC_LAYERS):
        for wire in range(N_QUBITS):
            theta, phi = np.asarray(noise_weights[layer, wire], dtype=float)
            circuit.ry(theta, wire)
            circuit.rz(phi, wire)
        for wire in range(N_QUBITS):
            circuit.cx(wire, (wire + 1) % N_QUBITS)
    circuit.measure(0, 0)
    noise_circuits.append(circuit)
    noise_reference.append(float((1.0 - np.asarray(vqc_circuit(
        pnp.array(noise_angles[position:position + 1], requires_grad=False), noise_weights
    ), dtype=float).reshape(-1)[0]) / 2.0))

ideal_backend = AerSimulator()
matched_state = Statevector.from_instruction(
    noise_circuits[0].remove_final_measurements(inplace=False)
)
matched_exact_probability = float(sum(
    probability for basis_state, probability in enumerate(matched_state.probabilities())
    if basis_state & 1
))
circuit_match_error = abs(matched_exact_probability - noise_reference[0])
assert circuit_match_error < 1e-10, (
    f'Qiskit/PennyLane circuit mismatch before noise benchmarking: {circuit_match_error}'
)
transpiled_circuits = transpile(
    noise_circuits, basis_gates=['rz', 'sx', 'x', 'cx'],
    optimization_level=1, seed_transpiler=SEED,
)
depths = [circuit.depth() for circuit in transpiled_circuits]
noise_model = NoiseModel()
noise_model.add_all_qubit_quantum_error(depolarizing_error(0.001, 1), ['sx', 'x'])
noise_model.add_all_qubit_quantum_error(depolarizing_error(0.01, 2), ['cx'])
ideal_counts = ideal_backend.run(
    transpiled_circuits, shots=VQC_SHOTS, seed_simulator=SEED
).result().get_counts()
noisy_backend = AerSimulator(noise_model=noise_model)
noisy_counts = noisy_backend.run(
    transpiled_circuits, shots=VQC_SHOTS, seed_simulator=SEED
).result().get_counts()
def measured_positive_probability(counts):
    total = sum(counts.values())
    return counts.get('1', 0) / total
ideal_probabilities = np.asarray([measured_positive_probability(c) for c in ideal_counts])
noisy_probabilities = np.asarray([measured_positive_probability(c) for c in noisy_counts])
noise_reference = np.asarray(noise_reference)
print('\nQiskit Aer circuit benchmark (synthetic noise, not hardware-calibrated):')
print('Transpiled depth min/median/max:', min(depths), float(np.median(depths)), max(depths))
print('PennyLane/Qiskit analytic circuit parity error:', f'{circuit_match_error:.2e}')
print('Mean absolute shot-sampled probability error, ideal vs PennyLane:',
      f'{np.mean(np.abs(ideal_probabilities - noise_reference)):.4f}')
print('Mean absolute shot-sampled probability error, noisy vs PennyLane:',
      f'{np.mean(np.abs(noisy_probabilities - noise_reference)):.4f}')
print('Mean absolute noise-induced probability shift:',
      f'{np.mean(np.abs(noisy_probabilities - ideal_probabilities)):.4f}',
      '| sample count:', sample_count, '| shots per circuit:', VQC_SHOTS)

quantum_run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S.%fZ')
quantum_config = {
    'architecture': '4-qubit PennyLane VQC: trainable RY/RZ layers with ring CNOT entanglers',
    'input_features': '4 fold-fit TruncatedSVD Morgan components + 11 RDKit descriptors, projected to 4 PCA components',
    'layers': VQC_LAYERS, 'epochs': VQC_EPOCHS,
    'sampling_ratios': ['1:5', '1:10'],
    'validation': 'same 5 scaffold-held-out folds; natural prevalence retained',
    'noise_benchmark': 'Qiskit Aer; 2048 shots; 0.001 single-qubit sx/x and 0.01 cx depolarizing error; illustrative only',
}
quantum_rows = []
for method, row in vqc_summary.iterrows():
    metrics = {
        metric: {'mean': float(vqc_cv.loc[vqc_cv.Method.eq(method), metric].mean()),
                 'std': float(vqc_cv.loc[vqc_cv.Method.eq(method), metric].std())}
        for metric in ('AP', 'Hits_at_50', 'Precision_at_50', 'Recall_at_50')
    }
    metrics['configuration'] = {**quantum_config, 'method': method}
    quantum_rows.append({
        'RunID': quantum_run_id, 'RecordType': 'model_evaluation',
        'RecordID': method, 'Status': 'evaluated_on_scaffold_held_out_folds',
        'LabelDataSHA256': label_sha256, 'CandidateDataSHA256': candidate_sha256,
        'ModelOrSelection': method, 'ParametersOrMetrics': json.dumps(metrics, sort_keys=True),
        'ArtifactPath': '', 'ArtifactSHA256': '', 'SubmissionReference': '',
        'Notes': 'Small classical simulation; not used to rank the full candidate library or submitted.',
    })
noise_metrics = {
    'transpiled_depth_min': int(min(depths)),
    'transpiled_depth_median': float(np.median(depths)),
    'transpiled_depth_max': int(max(depths)),
    'pennylane_qiskit_analytic_circuit_match_error': float(circuit_match_error),
    'sample_count': sample_count, 'shots_per_circuit': VQC_SHOTS,
    'mae_ideal_vs_analytic': float(np.mean(np.abs(ideal_probabilities - noise_reference))),
    'mae_noisy_vs_analytic': float(np.mean(np.abs(noisy_probabilities - noise_reference))),
    'mean_noise_shift': float(np.mean(np.abs(noisy_probabilities - ideal_probabilities))),
    'noise_rates_are_illustrative': True,
}
quantum_rows.append({
    'RunID': quantum_run_id, 'RecordType': 'noise_benchmark',
    'RecordID': 'qiskit_aer_synthetic_depolarizing_noise',
    'Status': 'simulated_on_fixed_validation_sample',
    'LabelDataSHA256': label_sha256, 'CandidateDataSHA256': candidate_sha256,
    'ModelOrSelection': quantum_config['architecture'],
    'ParametersOrMetrics': json.dumps(noise_metrics, sort_keys=True),
    'ArtifactPath': '', 'ArtifactSHA256': '', 'SubmissionReference': '',
    'Notes': 'Synthetic depolarizing rates are not hardware-calibrated and do not establish assay-score robustness.',
})
ledger = pd.read_csv(ledger_path, dtype=str, keep_default_na=False)
ledger = pd.concat([ledger, pd.DataFrame(quantum_rows)], ignore_index=True)
ledger.to_csv(ledger_path, index=False)
print('Quantum evaluation and noise benchmark appended to ledger:', quantum_run_id)
print('The full-candidate ranking remains the selected classical model; VQC was evaluated, not submitted.')


PennyLane VQC scaffold-held-out results (validation prevalence unchanged):
                      AP       Hits_at_50       Precision_at_50       Recall_at_50      
                    mean   std       mean   std            mean   std         mean   std
Method                                                                                  
PennyLane VQC 1:10 0.006 0.001      0.200 0.447           0.004 0.009        0.025 0.056
PennyLane VQC 1:5  0.029 0.050      0.400 0.894           0.008 0.018        0.050 0.112
PennyLane VQC 1:10: AP delta vs baseline=-0.415; hits@50 delta=-5.20
PennyLane VQC 1:5: AP delta vs baseline=-0.392; hits@50 delta=-5.00

Qiskit Aer circuit benchmark (synthetic noise, not hardware-calibrated):
Transpiled depth min/median/max: 17 17.0 17
PennyLane/Qiskit analytic circuit parity error: 1.67e-16
Mean absolute shot-sampled probability error, ideal vs PennyLane: 0.0101
Mean absolute shot-sampled probability error, noisy vs PennyLane: 0.0150
Mean absolute noise-in

## Submission discipline and provenance
The ledger records every completed model evaluation and ranking run; simply rerunning analysis does not count as a challenge submission. Do not submit these retrospective review files as prospective nominations. Before spending a submission opportunity, check the current challenge rules and template, freeze the model and selection criteria, and submit only a shortlist supported by the validation and practical checks. Do not change the model or repeatedly submit variants just to chase leaderboard feedback. After the platform accepts an actual submission and provides its official reference, run the next cell once with the exact uploaded file, reference, and source ranking record. It validates the source artifact, records the uploaded file's SHA-256, and refuses duplicate references. The helper never uploads or submits anything.

In [8]:
def record_challenge_submission(submitted_file, submission_reference, source_record_id,
                                *, submission_confirmed=False, notes=''):
    if submission_confirmed is not True:
        raise ValueError('Set submission_confirmed=True only after the challenge platform confirms the upload.')
    reference = str(submission_reference).strip()
    source_id = str(source_record_id).strip()
    if not reference:
        raise ValueError('Provide the official submission reference from the challenge platform.')
    if not source_id:
        raise ValueError('Provide the ledger RecordID of the ranking artifact used for this submission.')
    submitted_path = Path(submitted_file).expanduser().resolve()
    if not submitted_path.is_file():
        raise FileNotFoundError(f'Submitted file does not exist: {submitted_path}')
    if submitted_path.stat().st_size == 0:
        raise ValueError('Submitted file is empty; refusing to record it as a completed submission.')

    ledger = pd.read_csv(ledger_path, dtype=str, keep_default_na=False)
    if ledger.SubmissionReference.eq(reference).any():
        raise ValueError(f'Submission reference is already recorded: {reference}')
    sources = ledger.loc[
        ledger.RecordType.eq('ranking_artifact')
        & ledger.RecordID.eq(source_id)
        & ledger.Status.eq('generated_for_review')
    ]
    if sources.empty:
        raise ValueError(f'No generated ranking artifact is recorded with RecordID={source_id!r}.')
    source = sources.iloc[-1]
    source_path = DATA_DIR / source.ArtifactPath
    if not source_path.is_file() or sha256_file(source_path) != source.ArtifactSHA256:
        raise ValueError(f'Source ranking artifact is missing or has changed: {source_path}')

    try:
        logged_path = str(submitted_path.relative_to(DATA_DIR.resolve()))
    except ValueError:
        logged_path = str(submitted_path)
    row = {
        'RunID': datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S.%fZ'),
        'RecordType': 'submission',
        'RecordID': reference,
        'Status': 'submitted',
        'LabelDataSHA256': source.LabelDataSHA256,
        'CandidateDataSHA256': source.CandidateDataSHA256,
        'ModelOrSelection': source.ModelOrSelection,
        'ParametersOrMetrics': json.dumps({
            'source_record_id': source_id,
            'source_artifact_sha256': source.ArtifactSHA256,
        }, sort_keys=True),
        'ArtifactPath': logged_path,
        'ArtifactSHA256': sha256_file(submitted_path),
        'SubmissionReference': reference,
        'Notes': str(notes).strip(),
    }
    pd.concat([ledger, pd.DataFrame([row])], ignore_index=True).to_csv(ledger_path, index=False)
    print('Recorded confirmed submission:', reference)
    print('Uploaded artifact SHA256:', row['ArtifactSHA256'])

# After a confirmed platform upload, call explicitly, for example:
# record_challenge_submission(
#     submitted_file=DATA_DIR / 'my_submitted_file.csv',
#     submission_reference='official-platform-reference',
#     source_record_id='model_top50_review',
#     submission_confirmed=True,
#     notes='Prospective round; selected after scaffold validation and eligibility checks.'
# )


## Fixed L1 and orthogonal fingerprint benchmark

This fixed-configuration scaffold-held-out comparison tests L1-regularized Morgan logistic regression (C=1), MACCS-key logistic regression, hashed topological-torsion logistic regression, and percentile-rank blends of Morgan L2 with each alternate fingerprint. It uses the existing five scaffold folds, with all fitting confined to each training fold; the same folds have already informed earlier experiments, so treat this as an exploratory screen rather than an unbiased final estimate. No hidden labels are used and this section does not submit or overwrite the existing candidate ranking. A chemically diverse retrospective shortlist is not automatically challenge-eligible; verify the current prospective-stage template and constraints before any upload.

In [9]:
from rdkit.Chem import MACCSkeys, rdFingerprintGenerator
from sklearn.linear_model import LogisticRegression

def new_sparse_morgan_model():
    return LogisticRegression(
        C=1.0, l1_ratio=1.0, class_weight='balanced', solver='liblinear',
        max_iter=5000, random_state=SEED,
    )

def new_binary_fingerprint_model():
    return LogisticRegression(
        C=1.0, class_weight='balanced', solver='liblinear',
        max_iter=5000, random_state=SEED,
    )

orthogonal_configurations = {
    'Logistic baseline': '2048-bit chirality-aware Morgan radius-2; class-weighted L2 logistic; C=1.',
    'Morgan plus physicochemical logistic': 'Baseline Morgan bits plus 11 scaled RDKit descriptors; class-weighted L2 logistic; C=1.',
    'Morgan plus physicochemical ExtraTrees': 'Morgan plus 11 RDKit descriptors; ExtraTrees with 300 trees, min_samples_leaf=2, max_features=0.2, class_weight=balanced.',
    'Nearest-positive similarity': 'Maximum Morgan radius-2 Tanimoto similarity to training-fold positives.',
    '50/50 rank average': 'Mean percentile ranks of Morgan logistic and nearest-positive similarity.',
    'Morgan L1 logistic C=1': '2048-bit chirality-aware Morgan radius-2; class-weighted logistic with l1_ratio=1; liblinear; C=1.',
    'MACCS L2 logistic C=1': '167-bit RDKit MACCS keys; class-weighted logistic with default l1_ratio=0; C=1.',
    'Topological torsion L2 logistic C=1': '2048-bit RDKit hashed topological torsions; class-weighted logistic with default l1_ratio=0; C=1.',
    'Morgan L2 + MACCS percentile-rank fusion': 'Mean within-fold percentile ranks of Morgan L2 and MACCS L2 scores.',
    'Morgan L2 + topological torsion percentile-rank fusion': 'Mean within-fold percentile ranks of Morgan L2 and topological-torsion L2 scores.',
}

def evaluate_scaffold_models(model_specs, feature_matrices, similarity_fps=None):
    rows = []
    for fold, (train_idx, valid_idx) in enumerate(
        scaffold_cv.split(X, y, groups=scaffolds), start=1
    ):
        fold_scores = {}
        for name, (factory, feature_name) in model_specs.items():
            matrix = feature_matrices[feature_name]
            model = factory()
            model.fit(matrix[train_idx], y[train_idx])
            valid_matrix = matrix[valid_idx]
            if hasattr(model, 'decision_function'):
                fold_scores[name] = model.decision_function(valid_matrix)
            else:
                fold_scores[name] = model.predict_proba(valid_matrix)[:, 1]
        if similarity_fps is not None:
            positive_fps = [similarity_fps[i] for i in train_idx if y[i] == 1]
            fold_scores['Nearest-positive similarity'] = np.asarray([
                max(DataStructs.BulkTanimotoSimilarity(similarity_fps[i], positive_fps))
                for i in valid_idx
            ])
        if 'Logistic baseline' in fold_scores and 'Nearest-positive similarity' in fold_scores:
            base_rank = pd.Series(fold_scores['Logistic baseline']).rank(pct=True).to_numpy()
            sim_rank = pd.Series(fold_scores['Nearest-positive similarity']).rank(pct=True).to_numpy()
            fold_scores['50/50 rank average'] = (base_rank + sim_rank) / 2.0
        if 'Logistic baseline' in fold_scores:
            baseline_rank = pd.Series(fold_scores['Logistic baseline']).rank(pct=True).to_numpy()
            for alternate in ('MACCS L2 logistic C=1', 'Topological torsion L2 logistic C=1'):
                alternate_rank = pd.Series(fold_scores[alternate]).rank(pct=True).to_numpy()
                blend_name = ('Morgan L2 + MACCS percentile-rank fusion'
                              if alternate.startswith('MACCS') else
                              'Morgan L2 + topological torsion percentile-rank fusion')
                fold_scores[blend_name] = (baseline_rank + alternate_rank) / 2.0
        for name, scores in fold_scores.items():
            labels = y[valid_idx]
            top = np.argsort(scores)[::-1][:min(50, len(scores))]
            hits = int(labels[top].sum())
            positives = int(labels.sum())
            rows.append({
                'Fold': fold, 'Method': name,
                'AP': average_precision_score(labels, scores),
                'Hits_at_50': hits, 'Precision_at_50': hits / len(top),
                'Recall_at_50': hits / positives if positives else np.nan,
            })
    return pd.DataFrame(rows)

benchmark_models = {
    'Logistic baseline': (new_model, 'morgan'),
    'Morgan plus physicochemical logistic': (new_augmented_logistic, 'augmented'),
    'Morgan plus physicochemical ExtraTrees': (new_extra_trees, 'augmented'),
    'Morgan L1 logistic C=1': (new_sparse_morgan_model, 'morgan'),
    'MACCS L2 logistic C=1': (new_binary_fingerprint_model, 'maccs'),
    'Topological torsion L2 logistic C=1': (new_binary_fingerprint_model, 'torsion'),
}
benchmark_features = {
    'morgan': X, 'augmented': X_augmented,
    'maccs': maccs_X, 'torsion': torsion_X,
}
classical_cv = evaluate_scaffold_models(
    benchmark_models, benchmark_features, similarity_fps=labeled_fps
)
classical_summary = classical_cv.groupby('Method')[
    ['AP', 'Hits_at_50', 'Precision_at_50', 'Recall_at_50']
].agg(['mean', 'std'])
print('Unified classical scaffold-held-out comparison (mean and fold SD):')
print(classical_summary.to_string(float_format=lambda value: f'{value:.3f}'))
cv_means = classical_cv.groupby('Method')[['AP', 'Hits_at_50']].mean()
baseline_mean = cv_means.loc['Logistic baseline']
orthogonal_summary = classical_summary
print('Candidate ranking was already generated above using the fixed AP + hits@50 selection rule.')

orthogonal_run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S.%fZ')
orthogonal_ledger_rows = []
for method, values in orthogonal_summary.iterrows():
    metrics = {
        metric: {stat: float(values[(metric, stat)]) for stat in ('mean', 'std')}
        for metric in ('AP', 'Hits_at_50', 'Precision_at_50', 'Recall_at_50')
    }
    metrics['configuration'] = orthogonal_configurations[method]
    orthogonal_ledger_rows.append({
        'RunID': orthogonal_run_id, 'RecordType': 'model_evaluation',
        'RecordID': method, 'Status': 'evaluated_on_scaffold_held_out_folds',
        'LabelDataSHA256': label_sha256, 'CandidateDataSHA256': candidate_sha256,
        'ModelOrSelection': method,
        'ParametersOrMetrics': json.dumps(metrics, sort_keys=True),
        'ArtifactPath': '', 'ArtifactSHA256': '', 'SubmissionReference': '',
        'Notes': 'Fixed one-pass comparison; existing scaffold folds; exploratory and not submitted.',
    })
orthogonal_ledger = pd.read_csv(ledger_path, dtype=str, keep_default_na=False)
orthogonal_ledger = pd.concat(
    [orthogonal_ledger, pd.DataFrame(orthogonal_ledger_rows)], ignore_index=True
)
orthogonal_ledger.to_csv(ledger_path, index=False)
print('Comparison recorded in experiment ledger:', orthogonal_run_id)
print('Full-candidate ranking remains unchanged:', selected_model_name)


Unified classical scaffold-held-out comparison (mean and fold SD):
                                                          AP       Hits_at_50       Precision_at_50       Recall_at_50      
                                                        mean   std       mean   std            mean   std         mean   std
Method                                                                                                                      
50/50 rank average                                     0.372 0.204      4.800 1.643           0.096 0.033        0.621 0.229
Logistic baseline                                      0.421 0.199      5.400 1.517           0.108 0.030        0.693 0.188
MACCS L2 logistic C=1                                  0.046 0.032      2.000 1.225           0.040 0.024        0.254 0.150
Morgan L1 logistic C=1                                 0.298 0.139      4.000 1.225           0.080 0.024        0.514 0.156
Morgan L2 + MACCS percentile-rank fusion               0.2